In [1]:
import pandas as pd
import nltk 

# Predicting Authorship of the Disputed Federalist Papers

The Federalist Papers are a collection of **85 essays** written by James Madison, Alexander Hamilton, and John Jay under the collective pseudonym "Publius" to promote the ratification of the United States Constitution.

Authorship of most of the papers were revealed some years later by Hamilton, though his claim to authorshipt of 12 papers were disputed for nearly 200 years (studies generally agree that the disputed essays were written by James Madison.)

| Author | Papers |
| :- | -: | 
| Jay | 2, 3, 4, 5, 64
| Madison | 10, 14, 37-48
| Hamilton | 1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17, 21-36, 59, 60, 61, 65-85
| Hamilton and Madison | 18, 19, 20
| Disputed | 49-58, 62, 63

The goal of this problem is to train a classifier that predicts the author of the disputed papers.

In [2]:
# load Federalist papers data
url = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Practice/master/Data/papers.csv'
data = pd.read_csv(url)
data.head()

,paper,author
0,To the People of the State of New York: AFTE...,Hamilton
1,To the People of the State of New York: WHEN...,Jay
2,To the People of the State of New York: IT I...,Jay
3,To the People of the State of New York: MY L...,Jay
4,To the People of the State of New York: QUEE...,Jay


In [3]:
# Federalist paper No. 1
print(data.paper[0])

 To the People of the State of New York:  AFTER an unequivocal experience of the inefficacy of the subsisting federal government, you are called upon to deliberate on a new Constitution for the United States of America. The subject speaks its own importance; comprehending in its consequences nothing less than the existence of the UNION, the safety and welfare of the parts of which it is composed, the fate of an empire in many respects the most interesting in the world. It has been frequently remarked that it seems to have been reserved to the people of this country, by their conduct and example, to decide the important question, whether societies of men are really capable or not of establishing good government from reflection and choice, or whether they are forever destined to depend for their political constitutions on accident and force. If there be any truth in the remark, the crisis at which we are arrived may with propriety be regarded as the era in which that decision is to be ma

In [4]:
data.author.value_counts()

Hamilton            51
Madison             14
Disputed            12
Jay                  5
Hamilton+Madison     3
Name: author, dtype: int64

**Part 1 (text processing):** remove stop words and punctuations from the papers, and lemmatize them.

In [5]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet


from nltk.corpus import stopwords

stop_words = stopwords.words('english')



import string
punctuation = [punc for punc in string.punctuation]


lemmatizer = WordNetLemmatizer()



from nltk.tokenize import word_tokenize



In [6]:


# process parts of speech function
def process_pos(pos):
    if pos.startswith('J'): # adjectives
        return wordnet.ADJ
    elif pos.startswith('V'): # verbes
        return wordnet.VERB
    elif pos.startswith('N'): # nouns
        return wordnet.NOUN
    elif pos.startswith('R'): # adverbs
        return wordnet.ADV
    else:
        return wordnet.NOUN



In [7]:


for i in range(len(data)):
    text = data.loc[i,'paper']
    words = word_tokenize(text)
    words = [word.lower() for word in words]
    lemmatized_words = [lemmatizer.lemmatize(word, pos=process_pos(pos)) 
                        for word,pos in nltk.pos_tag(words) 
                        if word not in stop_words and word not in punctuation]
    data.loc[i,'processed_text'] = ' '.join(lemmatized_words)



In [8]:
data

,paper,author,processed_text
0,To the People of the State of New York: AFTE...,Hamilton,people state new york unequivocal experience i...
1,To the People of the State of New York: WHEN...,Jay,people state new york people america reflect c...
2,To the People of the State of New York: IT I...,Jay,people state new york new observation people c...
3,To the People of the State of New York: MY L...,Jay,people state new york last paper assign severa...
4,To the People of the State of New York: QUEE...,Jay,people state new york queen anne letter 1st ju...
...,...,...,...
80,To the People of the State of New York: LET ...,Hamilton,people state new york let u return partition j...
81,To the People of the State of New York: THE ...,Hamilton,people state new york erection new government ...
82,To the People of the State of New York: THE ...,Hamilton,people state new york objection plan conventio...
83,To the People of the State of New York: IN T...,Hamilton,people state new york course forego review con...


**Part 2: train-test split**

We'll use the papers written by Hamilton and Madion as the training set, and the disputed papers as the testing set.

In [9]:
data_train = data[data.author.isin(['Hamilton','Madison'])]
data_test = data[data.author=='Disputed']

Extract feature matrices X_train and X_test, and target vector y_train

In [10]:
X_train = data_train.processed_text
y_train = data_train.author
X_test = data_test.processed_text

**Part 3:** build a classification pipeline (count vectorizer + Naive Bayes model) that predicts the author of a paper.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split

In [12]:


pipe = Pipeline(steps=[
    ('vect', CountVectorizer()), 
    ('clf', MultinomialNB()) 
])

pipe.fit(X_train,y_train)

Pipeline(steps=[('vect', CountVectorizer()), ('clf', MultinomialNB())])

**Part 4:** Use a grid search to tune the pipeline hyperparameters

In [13]:
params_dic =  {'vect__max_features' : [2000,5000,7000,10000],
               'vect__min_df' : [5,25,50],
               'vect__max_df' : [1.0,0.9,0.8],
               'vect__ngram_range' : [(1,1), (1,2)],
               }

grid = GridSearchCV(pipe,
                    params_dic,
                    scoring='accuracy', 
                    cv=5, 
                    n_jobs=-1,
                    verbose=2)
grid.fit(X_train,y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
80 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
80 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/sklearn/pipeline.py", line 401, in fit
    Xt = self._fit(X, y, **fit_params_steps)
  File "/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/sklearn/pipeline.py", line 359, in _fit
    X, fitted_trans

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect', CountVectorizer()),
                                       ('clf', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'vect__max_df': [1.0, 0.9, 0.8],
                         'vect__max_features': [2000, 5000, 7000, 10000],
                         'vect__min_df': [5, 25, 50],
                         'vect__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy', verbose=2)

In [14]:


grid.best_params_



{'vect__max_df': 1.0,
 'vect__max_features': 2000,
 'vect__min_df': 5,
 'vect__ngram_range': (1, 2)}

In [15]:


best_pipe = grid.best_estimator_

# store the vocabulary of X_train
words = best_pipe['vect'].get_feature_names_out()



best_pipe['clf'].classes_



array(['Hamilton', 'Madison'], dtype='<U8')

**Part 5:** How does your classification model choose between Hamilton and Madison?

In [16]:
ham_word_count = best_pipe['clf'].feature_count_[0,:]
mad_word_count = best_pipe['clf'].feature_count_[1,:]

In [17]:
words = pd.DataFrame({'word' : words,
                      'ham' : ham_word_count, 
                      'mad' : mad_word_count}).set_index('word')
words.head()

,ham,mad
word,,
abandon,7.0,2.0
ability,12.0,0.0
able,44.0,13.0
abolish,13.0,8.0
abolition,5.0,2.0


In [18]:


# add 1 to the columns counts to avoid dividing by 0


for i in range(len(words)):
    words.iloc[i,0]+=1.0
    words.iloc[i,1]+=1.0






In [19]:
words.dtypes

ham    float64
mad    float64
dtype: object

In [20]:
madsum = words['mad'].sum()

In [21]:

# convert the counts into frequencies
words.ham = words.ham/words.ham.sum()
words.mad = words['mad']/madsum
words.head()



# ratios
words['ham_ratio'] = words.ham/words.mad
words['mad_ratio'] = words.mad/words.ham



words.sort_values(by='mad_ratio', ascending=False).head(20)

,ham,mad,ham_ratio,mad_ratio
word,,,,
judiciary department,0.000080,27.0,0.056237,17.781932
whilst,0.000040,13.0,0.058400,17.123342
article confederation,0.000100,29.0,0.065448,15.279290
relief,0.000040,8.0,0.094900,10.537441
exist congress,0.000040,8.0,0.094900,10.537441
although,0.000040,8.0,0.094900,10.537441
executive judiciary,0.000120,22.0,0.103527,9.659321
reform,0.000060,11.0,0.103527,9.659321
respectively,0.000040,7.0,0.108457,9.220261


In [22]:
words.sort_values(by='ham_ratio', ascending=False).head(20)

,ham,mad,ham_ratio,mad_ratio
word,,,,
upon,0.007482,8.0,17.698794,0.056501
intend,0.000702,1.0,13.285958,0.075267
enough,0.000682,1.0,12.906359,0.077481
kind,0.001725,3.0,10.881832,0.091896
readily,0.000502,1.0,9.489970,0.105374
station,0.000481,1.0,9.110371,0.109765
nomination,0.000481,1.0,9.110371,0.109765
commonly,0.000461,1.0,8.730772,0.114537
matter,0.001344,3.0,8.477706,0.117956


The model checks to see the if the language is more likely to appear in the work of Hamilton or Madison and then assigns the corresponding prediction.

**Part 6:** use your classifier to find who was the most likely author of the 12 disputed essays: Hamilton or Madison.

In [25]:
y_test_pred = best_pipe.predict(X_test)

In [26]:
y_test_pred

array(['Madison', 'Madison', 'Madison', 'Madison', 'Hamilton', 'Madison',
       'Madison', 'Hamilton', 'Madison', 'Madison', 'Madison', 'Madison'],
      dtype='<U8')

In [28]:
pd.Series(y_test_pred).value_counts()

Madison     10
Hamilton     2
dtype: int64

The classifer predicts that ten of the twelve disputed papers were authored by Madison, which agrees with the literature.